# Site EM Lab Assignment

## Introduction

In this lab you will implement the expectation maximization part of the MEME+
algorithm described by Bailey and Elkan in their technical report titled
["Fitting a Mixture Model by Expectation Maximization to Discover Motifs in BioPolymers."](https://cdn.aaai.org/ISMB/1994/ISMB94-004.pdf)

The outer loops, initialization, and the various "hacks" they added to the basic
EM model are provided for you, so you can focus on the EM part. However, it is
still the most code you've had to write so far, by a significant amount.


## Files

Add code your code to [assignment/siteEM.py](siteEM.py) everywhere there
is a comment YOUR CODE HERE. 

- test data is provided in [assignment/data](data)

- The Bailey and Elkan hacks, and some other convenience functions are in 
[assignment/utils](utils).

- [assignment/__main__.py](__main__.py) provides a module entry point
which allows you to run the code in the package as a script.

## Running the code in a notebook

To evaluate the cells below, make sure you choose the python interpreter specified 
in the .env. To select, use Cmd-P to pull up the vscode command menu and select 
"Python: Select Interpreter" and look for .venv.

## Data representations

### Input list of sequences

The sequences passed to the site EM algorithm are represented as lists of
nucleotides. The nucleotides are represented by the integers 0, 1, 2, 3 with
the following mapping:

```raw
A <-> 0
C <-> 1
G <-> 2
T <-> 3
```

You should use these as indices into the probability frequency matrix (PFM).

The total input is a list of sequences, where each sequence is a list of integers.

### Model parameters

The parameters related to the motif which are passed to the site EM algorithm are stored in a
[SequenceModel](https://cse587a.github.io/cse587Autils/SequenceObjects/API/SequenceModel.html)
object. Briefly, this object stores the prior probability (`site_prior`) of 
observing a motif in a sequence, and probability of observing a non-motif sequence
(`background_prior`), which is $1 -$ `site_prior`. A `SequenceModel` object also stores 
Position Frequency Matricies (PFMs) for the motif and background distributions.

- The motif PFM, stored in the attribute `site_base_probs` is represented as a
list of list where each sublist has length 4 with indicies corresponding to the
nucleotide mapping above. Each sublist represents a position (typically called a
column based on the way PFM are commonly visualized) in the motif and describes
the relative frequency of each nucleotide at that position.

- The background PFM, stored in the attribute `background_base_probs` is a
single list of length 4 where each index corresponds to the nucleotide mapping
above. This PFM describes the relative frequency of each nucleotide in the
background. This is assumed to be the same for all positions in the motif.

### Posteriors and erasers

- The purpose of the `e_step()` is to calculate the **posteriors**.
This is important: The posterior probability of a motif of length motif_length
starting at each position in a single input sequence is represented as a numpy 
array with length sequence_length - motif_length + 1. In the code, this is
sometimes called a `posteriors_row`. The posteriors for the 
entire input is a list of these. The kth posterior in a `posteriors_row` is the 
posterior probability that the sequence from k to k + motif_length - 1, inclusive,
was generated from the motif model. 
In this implementation, each posteriors row is normalized so the probabilities
some to one, corresponding to the assumption of one motif occurrence per input
sequence. Some of the Bailey and Elkan papers specify an alternative normalization
where the user supplies an expected number of motif occurrences in the entire
input and the posterior probabilities are normalized so they sum to that number
across the entire input.

- Erasers is a list of numpy arrays with the same dimensions as input. The 
entries are initialized to 1 and  may have any value between 0 and 1, inclusive. 
They are multiplicative factors that reduce the expected counts of letters. If 
the $k^{th}$ letter occurs in position $i+m-1$ of sequence $j$, it would 
normally contribute `posteriors[i, j]` to the expected count of letter $k$ in 
column $m$ of the PFM. With erasers, it will contribute `posteriors[i,
j] * erasers[i, j+m-1]`. Bailey and Elkan are not very clear about this and 
different papers suggest different ways of doing it. Eraser updates are calculated
for you but you will need to use the eraser values to modify the expected counts
as above.

### Return values
The top level function, `siteEM_initializer`, returns list of pairs, each pair
being the result of searching for one motif 
```
[(motif1_posteriors, motif1_sequence_model),
...
 (motifn_posteriors, motifn_sequence_model)]
```
The posteriors are provided for debugging convenience -- usually you will just
look at the sequence model. Even when only one motif is requested, it will be 
returned in a list containing only one pair.

The the sequence that is most likely to be output by a pfm model can be printed 
out by calling `consensus_pfm` from the utils directory.

In general, it is worth looking at the tests carefully to see which modules need
to be imported, how the functions in the codebase are called, and what they're 
expected to return before you start coding.

## Questions
Please answer the following questions in a new cell below the question. Don't start this part until your code passes all the tests provided. You may have to instrument your code a little to get these answers out. If so, retest to make sure your code still passes all the tests.

**When you are finished with this notebook, make a pdf by clicking on three dots at the top of the notebook and selecting** ***Export***. 
**Then commit your pdf to your repo and push, so that we can grade the questions on the pdf**

1. **Convergence.** Using the sequence file smallTest.fasta, how many iterations does it take make MM to converge with accuracy 0.01 and no effective limit on the number of iterations? For this question, we've provided the test code -- you just need to evaluate it and write some text to answer the question.

In [ ]:
from assignment.siteEM import siteEM_intializer

# Use direct path instead of importlib.resources
import os
fasta_path = os.path.join('data', 'smallTest.fasta')

siteEM_intializer(fasta_path, 4, 0.01,
                  max_iterations=1000,
                  accuracy=0.01,
                  num_motifs_to_find=1,
                  site_base_probs_seed=42)

**Answer 1**
Your answer goes here.

2. **Dependence on initialization.** Using the sequence file smallTest.fasta with motif width 4, max iterations 100, and accuracy 0.01, run siteEM_initializer 10 times in a loop with seed=0 to 9. How many different consensus motifs did you get? If you got more than one, how similar were they?

In [ ]:
from assignment.siteEM import siteEM_intializer
import os
from importlib import reload
import assignment as assign
reload(assign) # This ensures code is reloaded each time
from assignment.utils.consensus_pfm import consensus_pfm

fasta_path = os.path.join('data', 'smallTest.fasta')


**Answer 2**
Your answer goes here.


3. **Comparison to the online MEME on real promoters.** There is a very fancy and heavily engineered MEME implementation available online at [http://meme-suite.org/tools/meme](http://meme-suite.org/tools/meme). The file PACPlusSeqs.fasta contains segments of the promoters of yeast genes most which encode ribosomal proteins. They are admittedly carefully chose, and these segments (between 100 and 300 bp upstream of the start codon) are known to contain most of the instances of the PAC motif.

   3.1. First, run your code on the provided sequence file PACPlusSeqs.fasta in the directory Testing. Use motif width 8, accuracy 0.01, and maximum iterations 100, True for includeReverseStrand, and 3 for numMotifsToFind. This may take about 5 minutes to run. If it runs for more than 30 minutes either you have an ancient computer or there is something wrong with your implementation. Examine your 3 outputs, first looking at the consensus (most probable base in each position) and then using prettyPrintPFM to look at the actual numbers.

   3.2. Now go to the MEME web site, upload the sequence file, change expected frequency to "one occurrence per sequence", number of motifs to 3, and under "Advanced Options" select "Search given strand only" and min and max motif widths both 8. Submit your job and wait a couple of minutes for the results to come up.
   
   3.3. How do your top 3 motifs compare to the top 3 found by the online implementation? Please consider which of the motifs are similar to each other, how similar they are, and how similar motifs were ranked among the 3 output by MEME and the 3 output by your code.

In [ ]:
import os
from importlib import reload
import assignment as assign
reload(assign) # This ensures code is reloaded each time
from assignment.siteEM import siteEM_intializer
from assignment.utils.consensus_pfm import consensus_pfm

fasta_path = os.path.join('data', 'PACPlusSeqs.fasta')


**Answer 3**  
Your answer goes here.
